In [27]:
# imports
import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("Imports loaded successfully.")

Imports loaded successfully.


In [29]:
# Dataset paths
DATASET_DIR = Path("/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection")

TRAIN_CSV_PATH = DATASET_DIR / "stage_2_train.csv"
TRAIN_DICOM_DIR = DATASET_DIR / "stage_2_train"

print("Dataset directory exists:", DATASET_DIR.exists())
print("Train CSV exists:", TRAIN_CSV_PATH.exists())
print("Train DICOM directory exists:", TRAIN_DICOM_DIR.exists())

print("\nDataset directory contents:")
if DATASET_DIR.exists():
    for item in sorted(DATASET_DIR.iterdir()):
        print("-", item.name)

Dataset directory exists: True
Train CSV exists: True
Train DICOM directory exists: True

Dataset directory contents:
- stage_2_sample_submission.csv
- stage_2_test
- stage_2_train
- stage_2_train.csv


In [30]:
# Load CSV
train_df_long = pd.read_csv(TRAIN_CSV_PATH)

print("Loaded stage_2_train.csv successfully.")
print("Shape:", train_df_long.shape)
print("\nFirst 10 rows:")
display(train_df_long.head(10))

print("\nColumn names:", list(train_df_long.columns))

Loaded stage_2_train.csv successfully.
Shape: (4516842, 2)

First 10 rows:


,ID,Label
0,ID_12cadc6af_epidural,0
1,ID_12cadc6af_intraparenchymal,0
2,ID_12cadc6af_intraventricular,0
3,ID_12cadc6af_subarachnoid,0
4,ID_12cadc6af_subdural,0
5,ID_12cadc6af_any,0
6,ID_38fd7baa0_epidural,0
7,ID_38fd7baa0_intraparenchymal,0
8,ID_38fd7baa0_intraventricular,0
9,ID_38fd7baa0_subarachnoid,0



Column names: ['ID', 'Label']


In [31]:
# Inspect raw label format
print("Unique values in 'Label':", sorted(train_df_long["Label"].unique()))
print("Number of unique IDs in 'ID':", train_df_long["ID"].nunique())

print("\nSample ID strings:")
display(train_df_long["ID"].head(12))

Unique values in 'Label': [np.int64(0), np.int64(1)]
Number of unique IDs in 'ID': 4516818

Sample ID strings:


0             ID_12cadc6af_epidural
1     ID_12cadc6af_intraparenchymal
2     ID_12cadc6af_intraventricular
3         ID_12cadc6af_subarachnoid
4             ID_12cadc6af_subdural
5                  ID_12cadc6af_any
6             ID_38fd7baa0_epidural
7     ID_38fd7baa0_intraparenchymal
8     ID_38fd7baa0_intraventricular
9         ID_38fd7baa0_subarachnoid
10            ID_38fd7baa0_subdural
11                 ID_38fd7baa0_any
Name: ID, dtype: object

In [32]:
# Extract image_id and class_name
train_df_long[["image_id", "class_name"]] = train_df_long["ID"].str.rsplit("_", n=1, expand=True)

print("After splitting:")
display(train_df_long.head(10))

print("\nUnique class names found:")
print(sorted(train_df_long["class_name"].unique()))

After splitting:


,ID,Label,image_id,class_name
0,ID_12cadc6af_epidural,0,ID_12cadc6af,epidural
1,ID_12cadc6af_intraparenchymal,0,ID_12cadc6af,intraparenchymal
2,ID_12cadc6af_intraventricular,0,ID_12cadc6af,intraventricular
3,ID_12cadc6af_subarachnoid,0,ID_12cadc6af,subarachnoid
4,ID_12cadc6af_subdural,0,ID_12cadc6af,subdural
5,ID_12cadc6af_any,0,ID_12cadc6af,any
6,ID_38fd7baa0_epidural,0,ID_38fd7baa0,epidural
7,ID_38fd7baa0_intraparenchymal,0,ID_38fd7baa0,intraparenchymal
8,ID_38fd7baa0_intraventricular,0,ID_38fd7baa0,intraventricular
9,ID_38fd7baa0_subarachnoid,0,ID_38fd7baa0,subarachnoid



Unique class names found:
['any', 'epidural', 'intraparenchymal', 'intraventricular', 'subarachnoid', 'subdural']


In [33]:
# Sanity checks before pivot
print("Rows per class:")
display(train_df_long["class_name"].value_counts())

print("\nRows per image (should usually be 6):")
rows_per_image = train_df_long.groupby("image_id").size()
display(rows_per_image.value_counts().sort_index())

print("\nExample images and their row counts:")
display(rows_per_image.head(10))

Rows per class:


class_name
epidural            752807
intraparenchymal    752807
intraventricular    752807
subarachnoid        752807
subdural            752807
any                 752807
Name: count, dtype: int64


Rows per image (should usually be 6):


6     752799
12         4
Name: count, dtype: int64


Example images and their row counts:


image_id
ID_000012eaf    6
ID_000039fa0    6
ID_00005679d    6
ID_00008ce3c    6
ID_0000950d7    6
ID_0000aee4b    6
ID_0000ca2f6    6
ID_0000f1657    6
ID_000178e76    6
ID_00019828f    6
dtype: int64

In [34]:
# find duplicate keys
dup_counts = (
    train_df_long
    .groupby(["image_id", "class_name"])
    .size()
    .reset_index(name="count")
)

problem_dups = dup_counts[dup_counts["count"] > 1].sort_values("count", ascending=False)

print("Number of duplicated keys:", len(problem_dups))
display(problem_dups.head(20))

Number of duplicated keys: 24


,image_id,class_name,count
1282122,ID_489ae4179,any,2
1282123,ID_489ae4179,epidural,2
1282124,ID_489ae4179,intraparenchymal,2
1282125,ID_489ae4179,intraventricular,2
1282126,ID_489ae4179,subarachnoid,2
1282127,ID_489ae4179,subdural,2
2356320,ID_854fba667,any,2
2356321,ID_854fba667,epidural,2
2356322,ID_854fba667,intraparenchymal,2
2356323,ID_854fba667,intraventricular,2


In [36]:
# Convert long format to image-level labels

# Check for duplicates first
duplicate_rows = train_df_long.duplicated(subset=["image_id", "class_name"], keep=False)
duplicates_df = train_df_long.loc[duplicate_rows].sort_values(["image_id", "class_name"])

print("Number of duplicated (image_id, class_name) rows:", duplicates_df.shape[0])

if len(duplicates_df) > 0:
    print("\nSample duplicated rows:")
    display(duplicates_df.head(20))

# Aggregate duplicates safely
# For this dataset, labels should be 0/1, so max() is a practical way to resolve duplicates
train_df_wide = (
    train_df_long
    .groupby(["image_id", "class_name"], as_index=False)["Label"]
    .max()
    .pivot(index="image_id", columns="class_name", values="Label")
    .reset_index()
)

# Remove leftover column index name
train_df_wide.columns.name = None

# Standardize column order
target_columns = [
    "any",
    "epidural",
    "intraparenchymal",
    "intraventricular",
    "subarachnoid",
    "subdural"
]

# Check that all expected columns exist
missing_cols = [col for col in target_columns if col not in train_df_wide.columns]
print("Missing target columns:", missing_cols)

train_df_wide = train_df_wide[["image_id"] + target_columns]

print("Wide dataframe created successfully.")
print("Shape:", train_df_wide.shape)
display(train_df_wide.head())

Number of duplicated (image_id, class_name) rows: 48

Sample duplicated rows:


,ID,Label,image_id,class_name
3705311,ID_489ae4179_any,0,ID_489ae4179,any
3705317,ID_489ae4179_any,0,ID_489ae4179,any
3705306,ID_489ae4179_epidural,0,ID_489ae4179,epidural
3705312,ID_489ae4179_epidural,0,ID_489ae4179,epidural
3705307,ID_489ae4179_intraparenchymal,0,ID_489ae4179,intraparenchymal
3705313,ID_489ae4179_intraparenchymal,0,ID_489ae4179,intraparenchymal
3705308,ID_489ae4179_intraventricular,0,ID_489ae4179,intraventricular
3705314,ID_489ae4179_intraventricular,0,ID_489ae4179,intraventricular
3705309,ID_489ae4179_subarachnoid,0,ID_489ae4179,subarachnoid
3705315,ID_489ae4179_subarachnoid,0,ID_489ae4179,subarachnoid


Missing target columns: []
Wide dataframe created successfully.
Shape: (752803, 7)


,image_id,any,epidural,intraparenchymal,intraventricular,subarachnoid,subdural
0,ID_000012eaf,0,0,0,0,0,0
1,ID_000039fa0,0,0,0,0,0,0
2,ID_00005679d,0,0,0,0,0,0
3,ID_00008ce3c,0,0,0,0,0,0
4,ID_0000950d7,0,0,0,0,0,0


In [37]:
# Post-pivot sanity checks
print("Missing values per column:")
display(train_df_wide.isna().sum())

print("\nUnique values in each target column:")
for col in target_columns:
    print(f"{col}: {sorted(train_df_wide[col].dropna().unique())}")

print("\nNumber of images:", len(train_df_wide))
print("Expected number of label columns:", len(target_columns))

Missing values per column:


image_id            0
any                 0
epidural            0
intraparenchymal    0
intraventricular    0
subarachnoid        0
subdural            0
dtype: int64


Unique values in each target column:
any: [np.int64(0), np.int64(1)]
epidural: [np.int64(0), np.int64(1)]
intraparenchymal: [np.int64(0), np.int64(1)]
intraventricular: [np.int64(0), np.int64(1)]
subarachnoid: [np.int64(0), np.int64(1)]
subdural: [np.int64(0), np.int64(1)]

Number of images: 752803
Expected number of label columns: 6


In [38]:
# Label summary
label_sums = train_df_wide[target_columns].sum().sort_values(ascending=False)
label_means = train_df_wide[target_columns].mean().sort_values(ascending=False)

summary_df = pd.DataFrame({
    "positive_count": label_sums.astype(int),
    "positive_fraction": label_means
})

print("Class distribution summary:")
display(summary_df)

Class distribution summary:


,positive_count,positive_fraction
any,107933,0.143375
subdural,47166,0.062654
intraparenchymal,36118,0.047978
subarachnoid,35675,0.047390
intraventricular,26205,0.034810
epidural,3145,0.004178


In [39]:
# Save reshaped labels
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

reshaped_csv_path = OUTPUT_DIR / "train_labels_wide.csv"
train_df_wide.to_csv(reshaped_csv_path, index=False)

print("Saved reshaped labels to:", reshaped_csv_path)
print("File exists:", reshaped_csv_path.exists())

Saved reshaped labels to: /kaggle/working/train_labels_wide.csv
File exists: True
